# `stability_flexibility_anatomy_dcc.py` — line-by-line walkthrough

This notebook is a **guided, runnable deconstruction of the DCC orchestration script**, not a replacement implementation. It follows the teaching style of `stability_flexibility_segregation_tutorial.ipynb`: inspect the production function, unpack every statement, run the real helper on synthetic data, inspect intermediate objects, and pause for predictions.

The production script has two arms:

1. **categorical:** S/F flags → four selectivity groups → coverage-conditioned group × anatomy test;
2. **continuous:** disjoint-half LWPC/LWPS scores → `delta = lwpc_s - lwps_s` → ROI and coordinate tests, reliability ceiling, leverage checks, and maps.

The last section explains dispatch (`categorical`, `continuous`, or `both`) and how the lightweight runner turns environment variables into `args`.

> **Safe defaults here:** synthetic data, a temporary output directory, few permutations, and no brain renderer. Increase permutations only after you understand the flow. The notebook calls production functions directly, so it stays honest about what the cluster job does.


## 0 · Setup and a source-code magnifying glass

The DCC module computes its project root from `__file__`, adds it to `sys.path`, forces Matplotlib's non-interactive `Agg` backend, and imports the analysis module as `sfa`. The notebook performs a location-independent root search because notebooks may launch with a different working directory.

`show_source` is our line-by-line tool: every major section below prints the **live production source with original file line numbers**. Read the numbered source beside the explanation rather than trusting a copied, potentially stale version.


In [ ]:
%matplotlib inline
import inspect
import os
import sys
import tempfile
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

root = Path.cwd().resolve()
while root != root.parent and not (root / "dcc_scripts" / "stats" / "stability_flexibility_anatomy_dcc.py").exists():
    root = root.parent
if not (root / "dcc_scripts" / "stats" / "stability_flexibility_anatomy_dcc.py").exists():
    raise RuntimeError("Launch this notebook from somewhere inside the GlobalLocal repository")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from dcc_scripts.stats import stability_flexibility_anatomy_dcc as dcc
from src.analysis.stats import stability_flexibility_anatomy as sfa

def show_source(obj):
    """Print live source with its original file line numbers."""
    lines, start = inspect.getsourcelines(obj)
    print("".join(f"{start+i:4d} | {line}" for i, line in enumerate(lines)))

print("repo:", root)
print("DCC module:", dcc.__file__)
print("analysis module:", sfa.__file__)
print("CONTRAST_MODE:", dcc.CONTRAST_MODE, "| EFFECT_MEASURE:", dcc.EFFECT_MEASURE)


### Imports and constants, decoded

- `FutureWarning` is silenced before imports to keep long jobs readable; it does **not** suppress errors.
- The `__file__`/`os.getcwd()` fallback supports scripts and interactive execution.
- `/hpc/home` adds the lab pipeline path only on the cluster.
- `matplotlib.use('Agg')` must happen before importing `pyplot`; it makes saving figures work without a display.
- `CONTRAST_MODE` may come from the environment, while `EFFECT_MEASURE` is deliberately fixed to Cohen's *d*.
- `_LEVEL_TO_COL` translates a user-facing choice (`group`/`destrieux`) into a dataframe column (`roi`/`anat`).

Run the next cell if you want the exact top-level statements. `inspect` cannot represent an arbitrary file slice, so it reads and numbers lines 83–116 directly.


In [ ]:
script_path = Path(dcc.__file__)
script_lines = script_path.read_text().splitlines()
for number in range(83, 117):
    print(f"{number:4d} | {script_lines[number-1]}")


## 1 · Build the same `args` contract as the runner

The DCC core intentionally takes one namespace instead of parsing CLI flags. `run_stability_flexibility_anatomy_dcc.py` translates environment variables into that namespace; Slurm only wraps the runner.

Below is a local teaching configuration. Every attribute accessed by either synthetic arm is explicit. `n_perm=199` is pedagogical, **not publication-grade**; `make_brain=False` avoids MNE/PyVista/fsaverage requirements.


In [ ]:
work = Path(tempfile.mkdtemp(prefix="anatomy_dcc_tutorial_"))
args = SimpleNamespace(
    data_source="synthetic", synthetic_enrichment=0.6,
    arm="categorical", label_source="a1",
    subjects=[], task="GlobalLocal", acc_trials_only=True,
    LAB_root=None, epochs_root_file=None,
    window_tmin=0.0, window_tmax=0.5,
    electrodes="all", rois_dict=None, roi_dict_dir=None,
    scores_csv=None, per_split_csv=None, n_splits=20,
    responsiveness=None, use_coords=True,
    pt_runs=None, pt_correction="fdr_bh", pt_alpha=None,
    pt_roi=None, pt_require_all=True,
    roi_filter=None, anat_level="auto", hist_top_n=12,
    make_brain=False, brain_hemi="both", brain_subjects=None,
    alpha=0.05, fdr_correction="fdr_bh",
    min_subjects=3, n_perm=199, seed=0,
    save_dir=str(work / "categorical"),
)
print("tutorial outputs:", work)
vars(args)


## 2 · `resolve_anat_level`: prevent a vacuous one-column test


In [ ]:
show_source(dcc.resolve_anat_level)


Line by line:

1. If the caller explicitly asks for `group` or `destrieux`, dictionary lookup returns `roi` or `anat` immediately.
2. Anything other than those values or `auto` is rejected early—typos must not silently change the scientific question.
3. `single_roi` becomes true for an explicit ROI filter **or** when the attached table happens to contain at most one non-null coarse ROI.
4. Fine anatomy is chosen only if usable `anat` values actually exist.
5. Otherwise the safe whole-brain default is the coarse `roi` column.

The following tiny examples exercise all branches.


In [ ]:
toy = pd.DataFrame({"roi": ["lpfc", "lpfc"], "anat": ["A", "B"]})
print("auto, one ROI + anatomy:", dcc.resolve_anat_level("auto", toy))
print("explicit group:", dcc.resolve_anat_level("group", toy))
print("explicit destrieux:", dcc.resolve_anat_level("destrieux", toy))
try:
    dcc.resolve_anat_level("lobes", toy)
except ValueError as exc:
    print("expected guard:", exc)


# Part I — categorical arm

## 3 · Where labels come from

`main_categorical` supports three routes:

- synthetic ground truth for path validation;
- real A1 labels computed from epochs (`load_a1_labels`);
- labels read from finished windowed `power_traces` runs (`load_power_traces_labels`).

We start with the synthetic branch because it has the same downstream table contract and needs no protected data.


In [ ]:
labels, e2r, e2a = sfa._synthetic_anatomy(
    enrichment=args.synthetic_enrichment, seed=args.seed, return_anat=True)
print(labels.shape, "labels")
print(len(e2r), "coarse mappings |", len(e2a), "Destrieux mappings")
display(labels.head())


### Real A1 route: `load_a1_labels`, line by line


In [ ]:
show_source(dcc.load_a1_labels)


- Imports are lazy: the synthetic and `power_traces` routes do not pay for epoch-loading dependencies.
- `load_HG_ev1_rescaled_per_subject` loads each subject's data.
- `resolve_electrodes_to_keep` applies the common `all`/`sig` selection.
- `assemble_long_df` collapses the requested time window into the long trial table.
- Both proportion columns are mandatory because A1 is defined as the LWPC/LWPS interaction; missing columns raise rather than silently switching estimands.
- The long table is saved for provenance, then Type III per-electrode ANOVAs are FDR-corrected.


### Finished `power_traces` route: `load_power_traces_labels`, line by line


In [ ]:
show_source(dcc.load_power_traces_labels)


- `runs` may be one four-factor run directory or a mapping of effect names to directories.
- `pt_alpha` overrides the main alpha only when supplied.
- `electrode_labels` produces the same `subject, electrode, S, F` contract as A1.
- `require_all` controls whether electrodes missing from a component run are dropped.
- `attrs['label_funnel']` records attrition diagnostics; it is printed now and saved later.


## 4 · Anatomy maps and the overlap trap


In [ ]:
show_source(dcc.load_anatomy_maps)


The atlas is nested by subject/channel. This helper returns two flat maps: coarse ROI and raw Destrieux label. The subtle line is `subset_rois_dict(..., roi_filter)` **before** building the coarse map. Coarse groups overlap, and mapping is first-group-wins; filtering afterward could let an earlier group steal labels from the requested group.

Our synthetic generator already returned equivalent maps, so attach them now.


In [ ]:
lab_roi = sfa.attach_roi(labels, e2r, electrodes_to_anat=e2a)
display(lab_roi[["subject", "electrode", "S", "F", "group", "roi", "anat"]].head())
print("group counts:\n", lab_roi["group"].value_counts())
print("unmapped coarse ROI:", lab_roi["roi"].isna().sum())


## 5 · Restriction, level choice, and coverage

Order matters: labels are defined first, anatomy is attached second, restriction happens third, and coverage is recomputed on the final anatomical population. Restriction must never redefine S/F.


In [ ]:
restricted = sfa.restrict_to_roi(lab_roi, args.roi_filter)
roi_col = dcc.resolve_anat_level(args.anat_level, restricted, args.roi_filter)
coverage = sfa.build_coverage_matrix(restricted, roi_col=roi_col)
print("testing column:", roi_col)
print("coverage shape:", coverage.shape)
display(coverage.astype(int).head())


**Prediction pause.** Why is coverage built from all mapped electrodes, including `neither`? Because coverage describes where surgeons placed contacts, not where effects passed a threshold. Using selective electrodes would contaminate the conditioning variable with the outcome.

## 6 · Coverage-conditioned enrichment

The observed statistic is group × ROI χ². The null permutes group labels **within subject**, holding both physical ROI placements and each subject's group counts fixed.


In [ ]:
enrich = sfa.roi_group_enrichment_test(
    restricted, coverage, min_subjects=args.min_subjects,
    roi_col=roi_col, n_perm=args.n_perm, seed=args.seed)
print("ROIs tested:", enrich["rois_tested"])
print("chi2:", enrich["observed_stat"], "| p:", enrich["p"])
print("electrodes:", enrich["n_electrodes"])
display(enrich["contingency"])


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(enrich["null"], bins=25, color="0.75")
ax.axvline(enrich["observed_stat"], color="crimson", lw=2, label="observed")
ax.set(xlabel="within-subject null chi-square", ylabel="count")
ax.legend(); plt.show()


## 7 · `save_results`: durable tables before pictures


In [ ]:
show_source(dcc.save_results)


The function creates the directory, then saves:

1. every labeled electrode with anatomy;
2. the boolean coverage matrix as 0/1;
3. the tested contingency table;
4. histograms at coarse and, when available, Destrieux levels;
5. compact JSON metadata without the large null array;
6. the permutation null separately as NumPy binary data.

Separating compact metadata and the full null keeps results human-readable without losing reproducibility.


In [ ]:
dcc.save_results(restricted, coverage, enrich, args.save_dir, roi_col=roi_col)
for path in sorted(Path(args.save_dir).iterdir()):
    print(path.name, f"({path.stat().st_size:,} bytes)")


## 8 · `make_plots`: three figure families and a guarded brain renderer


In [ ]:
show_source(dcc.make_plots)


- The coarse histogram is always made; the Destrieux histogram is added when possible.
- Coverage and the permutation null share a diagnostic figure so sampling and inference stay visually linked.
- `plt.close` prevents memory accumulation in batch jobs.
- The brain renderer is optional and handles its own fallback. Here it is disabled deliberately.


In [ ]:
brain = dcc.make_plots(
    restricted, coverage, enrich, args.save_dir, roi_col=roi_col,
    subjects=None, hemi="both", make_brain=False, hist_top_n=12)
print("brain result:", brain)
print("PNGs:", [p.name for p in sorted(Path(args.save_dir).glob("*.png"))])


## 9 · `write_summary`: turn outputs into an auditable narrative


In [ ]:
show_source(dcc.write_summary)


This function does no inference. It formats the already-computed results: mapping counts, group counts, tested anatomy level, coverage, statistic/p-value, contingency table, optional raw-label counts, and brain-render status. The `meta` mapping records knobs alongside conclusions.


In [ ]:
dcc.write_summary(
    restricted, coverage, enrich, args.save_dir,
    meta={"data_source": "synthetic", "tutorial": True},
    min_subjects=args.min_subjects, alpha=args.alpha,
    roi_col=roi_col, brain=brain)
print("\n--- saved summary ---")
print((Path(args.save_dir) / "summary.txt").read_text()[:1800])


## 10 · Read the categorical orchestrator end to end


In [ ]:
show_source(dcc.main_categorical)


Map each source block to what we just executed:

1. validate settings and resolve a lab root only when truly needed;
2. obtain labels and two anatomy maps;
3. attach anatomy, then optionally restrict;
4. resolve anatomy level and build coverage;
5. run the within-subject permutation test;
6. save, plot, summarize, and return the in-memory objects.

Now run that exact orchestrator in a fresh directory. This is the production path, merely with synthetic input and a small permutation count.


In [ ]:
args_cat = SimpleNamespace(**vars(args))
args_cat.save_dir = str(work / "categorical_end_to_end")
out_cat = dcc.main_categorical(args_cat)
print("returned keys:", out_cat.keys())
print("saved files:", sorted(p.name for p in Path(args_cat.save_dir).iterdir()))


# Part II — continuous arm

The continuous arm avoids thresholding electrodes. Each electrode has disjoint-half LWPC and LWPS scores; anatomy is tested using their within-electrode difference:

\[
\Delta = \mathrm{LWPC}_s - \mathrm{LWPS}_s.
\]

Positive Δ means LWPC-dominant. Swapping the two effect labels within an electrode flips Δ and supplies the null while holding anatomy, subject, and electrode quality fixed.

## 11 · `load_scores`: cached CSV or expensive recomputation


In [ ]:
show_source(dcc.load_scores)


Line by line:

- A supplied `scores_csv` is the cheapest route; `per_split_csv` is optional computationally but required for the reliability ceiling and `min_elec` sensitivity analysis.
- Otherwise the helper lazily loads epochs, applies electrode selection, assembles the long table, and computes sensitivities on disjoint halves.
- `average_over_splits` produces electrode scores; `add_responsiveness` adds the gain/SNR covariate.
- The caller saves both tables so future jobs can take the fast route.

The synthetic arm bypasses `load_scores` but creates the **same contracts**, including fake split-level estimates so every diagnostic path is exercised.


In [ ]:
scores, e2r_s, e2a_s, e2c_s = sfa._synthetic_scores(
    gradient=args.synthetic_enrichment, seed=args.seed)
per_split = sfa._synthetic_per_split(scores, n_splits=20, seed=args.seed)
print("scores:", scores.shape, "| per split:", per_split.shape)
display(scores.head())
display(per_split.head())


## 12 · Attach scores, anatomy, and coordinates


In [ ]:
score_tab = sfa.attach_scores(
    scores, e2r_s, electrodes_to_anat=e2a_s, electrodes_to_coords=e2c_s)
score_tab = sfa.restrict_to_roi(score_tab, args.roi_filter)
score_roi_col = dcc.resolve_anat_level(args.anat_level, score_tab, args.roi_filter)
score_coverage = sfa.build_coverage_matrix(score_tab, roi_col=score_roi_col)
cols = ["subject", "electrode", "lwpc_s", "lwps_s", "delta", "resp",
        "roi", "anat", "mni_x", "mni_y", "mni_z", "hemi"]
display(score_tab[cols].head())
print("coverage:", score_coverage.shape, "at", score_roi_col)


## 13 · Primary ROI test and leave-one-subject-out leverage

The primary model asks whether Δ varies by covered ROI after accounting for responsiveness and subject. The permutation swaps effect identity within electrodes. LOSO reruns the same statistic with each subject removed; it is a leverage check, not a second inferential family.


In [ ]:
roi_res = sfa.relative_score_roi_test(
    score_tab, score_coverage, min_subjects=args.min_subjects,
    n_perm=args.n_perm, seed=args.seed, roi_col=score_roi_col)
print("F:", roi_res["observed_stat"], "p:", roi_res["p"])
display(roi_res["per_roi"])


In [ ]:
loso = sfa.leave_one_subject_out(
    lambda table: sfa.relative_score_roi_test(
        table, score_coverage, min_subjects=args.min_subjects,
        roi_col=score_roi_col, n_perm=99, seed=args.seed),
    score_tab)
display(loso.head())
print("folds:", len(loso), "| statistic range:",
      (loso.observed_stat.min(), loso.observed_stat.max()))


## 14 · Secondary coordinate model and descriptive centers

When MNI coordinates exist, the script runs a hemisphere-specific model of Δ against y/z/x, responsiveness, and subject. Positive `mni_y` means more anterior LWPC dominance. Weighted centers are descriptive and are explicitly not promoted over the regression.


In [ ]:
coord_res = sfa.relative_score_coordinate_test(score_tab, n_perm=99, seed=args.seed)
for hemi, result in coord_res.items():
    print("\n", hemi, "F=", result["observed_stat"], "p=", result["p"])
    display(result["slopes"])

centers = sfa.score_centers_per_subject(score_tab, n_perm=99, seed=args.seed)
print("center groups:", centers.get("n_groups"))
print("mean displacement:", centers.get("mean_displacement"))


## 15 · Noise ceiling, `min_elec` sweep, and scatter diagnostics

A low LWPC–LWPS spatial correlation is meaningful only when each map is reliable. `map_reliability` estimates split-half reliability at electrode and parcel levels and reports a noise-corrected between-map correlation.

The production orchestrator also sweeps `min_elec = 1,2,3`. That parameter drops whole subjects, so it can materially change effective N. We show one lightweight call here; the exact orchestrator performs the complete sweep with 2,000 permutations per row.


In [ ]:
from src.analysis.stats import stability_flexibility_segregation as sfs
from src.analysis.stats import segregation_scatter as scat

parcels = dict(zip(score_tab.electrode.astype(str), score_tab[score_roi_col]))
electrode_ceiling = sfa.map_reliability(per_split)
parcel_ceiling = sfa.map_reliability(per_split, parcels=parcels)
print("electrode ceiling:", electrode_ceiling)
print("parcel ceiling:", parcel_ceiling)

resp = score_tab.drop_duplicates("electrode").set_index("electrode")["resp"]
quick_corr = sfs.split_resolved_corr(per_split, resp, min_elec=2,
                                     n_perm=99, seed=args.seed)
print("quick min_elec=2 check:", quick_corr)


In [ ]:
scatter_diag = scat.joint_scatter_diagnostics(
    score_tab, value_cols=("lwpc_s", "lwps_s"))
scatter_diag


## 16 · Figures and outputs in the continuous arm

The arm writes a joint scatter (with the ceiling annotation), Δ-by-ROI plot, optionally five brain maps, score/anatomy and coverage tables, per-ROI results, LOSO results, centers, a min-electrode sweep, JSON, and a top-to-bottom text summary.

Read the exact orchestrator now; its numbered source is the definitive ordering.


In [ ]:
show_source(dcc.run_score_anatomy)


### Statement-by-statement map of `run_score_anatomy`

1. Lazily import scatter plotting; normalize knobs; make `save_dir/continuous`.
2. Choose synthetic scores or real/cached scores and load anatomy/coordinates.
3. Persist raw score inputs immediately.
4. Attach maps, restrict ROI, guard against an empty population, choose anatomical resolution, and build coverage.
5. Run the primary relative-score ROI test.
6. Run LOSO with fewer permutations because it diagnoses leverage rather than creating another primary p-value.
7. If coordinates exist, run coordinate regression and centers; otherwise state the skip.
8. If split-level data exist, compute electrode/parcel reliability and the three-row `min_elec` sweep. Exceptions become table notes rather than destroying all other outputs.
9. Compute pooled and within-subject scatter diagnostics.
10. Render/save figures, close Matplotlib state, and write all tables.
11. Construct compact JSON, write the narrative summary, and return rich in-memory results.

Run the real function next. It does more work than the hand-held cells because the production `min_elec` sweep uses 2,000 permutations.


In [ ]:
args_cont = SimpleNamespace(**vars(args))
args_cont.arm = "continuous"
args_cont.save_dir = str(work / "continuous_end_to_end")
out_cont = dcc.run_score_anatomy(args_cont)
print("returned keys:", out_cont.keys())
print("continuous directory:", out_cont["save_dir"])
print("saved files:", sorted(p.name for p in Path(out_cont["save_dir"]).iterdir()))


## 17 · `write_score_summary`, line by line


In [ ]:
show_source(dcc.write_score_summary)


The summary deliberately teaches the reading order:

- define scores and Δ;
- state the primary covered-ROI result;
- examine LOSO changes in the **statistic**, not only saturated permutation p-values;
- show secondary coordinate slopes and descriptive centers;
- refuse to overinterpret map correlation without its ceiling;
- compare pooled and within-subject score correlations;
- end by naming the estimand: an effect-type × anatomy interaction.

Inspect the artifact produced above.


In [ ]:
summary_path = Path(out_cont["save_dir"]) / "summary.txt"
print(summary_path.read_text()[:4000])


# Part III — dispatch, runner, and your own modifications

## 18 · `main`: the complete dispatcher


In [ ]:
show_source(dcc.main)


Only three values are legal. `continuous` returns early, `categorical` runs alone, and `both` runs categorical then nests continuous output under `out['continuous']`. The early return prevents accidental categorical work when only scores were requested.


In [ ]:
bad = SimpleNamespace(**vars(args))
bad.arm = "mystery"
try:
    dcc.main(bad)
except ValueError as exc:
    print("expected guard:", exc)


## 19 · How the runner fills `args`

Open `dcc_scripts/stats/run_stability_flexibility_anatomy_dcc.py` after this notebook. Its responsibilities are deliberately separate:

1. read environment variables;
2. validate combinations that can be checked before loading data;
3. parse comma-separated ROI filters;
4. create a descriptive output path;
5. build `SimpleNamespace`;
6. call `main(args)`.

The core never reaches into runner globals. This separation makes the synthetic notebook and unit tests possible.

Typical cluster calls:

```bash
# fast path validation
DATA_SOURCE=synthetic ARM=both MAKE_BRAIN=0 N_PERM=1000   bash dcc_scripts/stats/submit_stability_flexibility_anatomy_dcc.sh

# categorical from finished time-resolved ANOVA results
DATA_SOURCE=real LABEL_SOURCE=power_traces PT_RUN_DIR=/path/to/run   ROI_FILTER=lpfc ANAT_LEVEL=auto   bash dcc_scripts/stats/submit_stability_flexibility_anatomy_dcc.sh

# continuous from cached segregation scores + required ceiling table
ARM=continuous SCORES_CSV=/path/to/electrodes.csv   PER_SPLIT_CSV=/path/to/per_split.csv   bash dcc_scripts/stats/submit_stability_flexibility_anatomy_dcc.sh
```

Do not treat the tutorial's 199 permutations as analysis settings.


## 20 · Output checklist: trace every artifact back to its creator


In [ ]:
for path in sorted(work.rglob("*")):
    if path.is_file():
        print(path.relative_to(work), f"{path.stat().st_size:,} bytes")


Use this provenance map when debugging:

| Artifact | Created by | Meaning |
|---|---|---|
| `anatomy_labels_roi.csv` | `save_results` | categorical labels plus both anatomy levels |
| `coverage_matrix.csv` | both arms | subject × tested-anatomy coverage |
| `roi_enrichment.json` / `roi_enrichment_null.npy` | `save_results` | categorical statistic metadata / full null |
| `scores.csv`, `per_split.csv` | `run_score_anatomy` | continuous inputs and reliability substrate |
| `scores_with_anatomy.csv` | `run_score_anatomy` | score table used in spatial tests |
| `delta_per_roi.csv` | `run_score_anatomy` | adjusted ROI estimates |
| `delta_roi_loso.csv` | `run_score_anatomy` | leverage audit |
| `min_elec_sweep.csv` | `run_score_anatomy` | whole-subject inclusion sensitivity |
| `score_anatomy.json` | `run_score_anatomy` | compact machine-readable result |
| `summary.txt` | summary writers | human-readable methods/results audit |

## 🧠 Final self-check

Before revealing these answers, explain each in your own words.

<details><summary>Why attach labels before restricting anatomy?</summary>
The electrode definition must remain independent of the anatomical question. Filtering trials/electrodes before defining S/F would change the population and potentially the multiple-testing family.
</details>

<details><summary>Why permute categorical group within subject?</summary>
It preserves the actual placements and each subject's group composition, breaking only their pairing. This conditions the null on clinical coverage and subject nesting.
</details>

<details><summary>Why does the continuous null swap LWPC/LWPS within electrode?</summary>
The tested response is their paired difference. Swapping labels flips the effect-type contrast while leaving electrode anatomy, subject, responsiveness, and measurement quality fixed.
</details>

<details><summary>Why is a per-split file scientifically important?</summary>
It supports the reliability ceiling. Without it, weak cross-map correlation cannot distinguish genuinely different maps from unreliably estimated maps.
</details>

<details><summary>Why inspect LOSO statistics rather than only p-values?</summary>
Permutation p-values can saturate at their resolution floor. Changes in F reveal whether one subject carries the effect even when every fold prints the same small p-value.
</details>


## 🏁 Capstone exercises

1. Set `roi_filter='lpfc'` and predict why `auto` changes from `roi` to `anat`; rerun each arm.
2. Set `synthetic_enrichment=0.0`; verify the categorical test does not manufacture enrichment and the continuous spatial gradient weakens.
3. Delete `per_split` conceptually (or load scores without `PER_SPLIT_CSV`) and list exactly which outputs disappear.
4. Compare `main_categorical(args)` with `main(args)` for `arm='categorical'`; confirm their return contracts match.
5. Increase `min_subjects` and record how tested regions, electrodes, subjects, and the statistic change.
6. Only after all of the above, switch to real inputs and publication-scale permutations.

The core habit is: **stop after every transformation, inspect its contract, and state which nuisance variables the next test preserves.**
